# Classes, Properties, and App Data

Model related values and behavior with Kotlin classes, data classes, and interfaces.

Functions gave our app rules names. Classes now give related data a shared structure. We will build on your Java and C++ object-oriented experience while focusing on Kotlin's shorter declarations and explicit rules for changing and extending objects.

## Learning Goals

- Create objects with constructor properties and controlled access to their state.
- Create a modified data-class copy and explain what remains unchanged.
- Implement an interface and recognize Kotlin's explicit inheritance and override rules.

## Why This Matters

An app manages related facts: a note has a title and an archive state, while a reminder has a title and a completion state. Keeping each fact in an unrelated variable makes it easier to mix up which values belong together.

A class groups those values into a model: a representation of something the app manages. It can also keep changes behind named operations. Interfaces let different implementations promise the same useful behavior. These choices help separate data rules from screens, so you can test the model before building an Android interface.

## Check Your Starting Point

Lesson 2 introduced default arguments. A function declares `fallback: String = "Guest"`. Explain which value is used when a caller omits `fallback`, and which is used when the caller supplies `fallback = "Visitor"`. Why does supplying a default not make the parameter nullable?

In [ ]:
Your response:
Write your explanation here.

<details>
<summary>Show answer</summary>

Omitting the argument uses `Guest`. Supplying `Visitor` uses that caller's value for this call. The type is still `String`, which requires non-null text. Optional at the call site and nullable are different ideas. Kotlin constructors can use the same default-argument pattern.

</details>

## Video Demonstration

Watch related note data become a data class, then follow a modified copy and an interface implementation. The original object remains available for comparison.

<video style="max-width:100%;height:auto;" controls preload="metadata" width="800" aria-label="Classes, Properties, and App Data demonstration">
<source src="media/03_classes_properties_and_app_data/lesson.mp4" type="video/mp4">
<track kind="captions" src="media/03_classes_properties_and_app_data/captions.vtt" srclang="en" label="English">
Your browser does not support embedded video.
</video>

[Read the video transcript and visual description](media/03_classes_properties_and_app_data/transcript.md).

## Concept

### Construct an Object and Expose Properties

A **class** defines the structure and behavior of its **objects**, the individual instances made from it. Kotlin's `class` keyword introduces the declaration. A **primary constructor** places the main construction parameters immediately after the class name.

```kotlin
class NoteDraft(val title: String, var archived: Boolean = false)
```

This one line defines the class and two **properties**, named values that belong to each object. `val title` creates a read-only property; `var archived` creates a property that can be reassigned. The default `false` means a caller can create an unarchived draft by supplying only a title.

Call `NoteDraft("Travel")` to create an object. Kotlin does not use Java's `new` keyword here. A dot accesses an object's property, such as `draft.archived`. A constructor parameter becomes a property when it is declared with `val` or `var`; a plain parameter can instead be used while initializing the object without becoming a public property.

In [ ]:
class NoteDraft(val title: String, var archived: Boolean = false)
val draft = NoteDraft("Travel")
println(draft.archived)
draft.archived = true
println("${draft.title}: ${draft.archived}")

This prints `false`, then `Travel: true`. The `draft` binding still refers to the same object; its `var` property changed. This is why a `val` binding does not guarantee that every part of its object is immutable. Reassigning `draft.title` would fail because that property is a `val`.

### Keep a Change Behind a Method

A **method** is a function declared inside a class. **Visibility** controls where a declaration can be accessed. Kotlin declarations are public by default. The `private` keyword restricts a class member to that class, which lets you expose operations instead of unrestricted changes.

```kotlin
class RevisionCounter(private var count: Int = 0) {
    fun bump(): Int {
        count += 1
        return count
    }
}
```

The constructor creates private state. The `bump` method increments it and returns the new value. Outside code can call `bump`, but cannot assign `counter.count` directly. This small boundary gives the class control over how its count changes.

In [ ]:
class RevisionCounter(private var count: Int = 0) {
    fun bump(): Int {
        count += 1
        return count
    }
}
val counter = RevisionCounter()
println(counter.bump())
println(counter.bump())

The results are `1` and `2`. Both calls act on the same counter. `private` is an access rule enforced by the language; it is not encryption or a security boundary against someone who controls the program.

### Run Initialization Work

An **initializer block**, introduced by `init`, runs during object construction. Use it when initialization needs statements beyond assigning constructor properties. Multiple property initializers and `init` blocks run in their order in the class body.

The next class replaces empty initial text with `Untitled`, then prints a construction message. An `init` block can update a `var` property with the same `if` and assignment syntax you already know. The print is a small trace to make the timing visible; an app model usually would not print simply because it was created.

In [ ]:
class DisplayTitle(var text: String) {
    init {
        if (text == "") { text = "Untitled" }
        println("Created: $text")
    }
}
val displayTitle = DisplayTitle("Travel")
println(displayTitle.text)
val emptyDisplayTitle = DisplayTitle("")
println(emptyDisplayTitle.text)

The first object prints `Created: Travel`, then its property prints `Travel`. The empty input produces `Created: Untitled`, then `Untitled`. The `init` block runs as part of each constructor call, before the following `println`. Reading the property afterward does not run initialization again. This rule normalizes the initial value; a public `var` could still be changed later.

### Describe Values with a Data Class

A **data class**, declared with `data class`, asks Kotlin to generate common value-oriented operations from properties in its primary constructor. These include useful text output, equality comparison, and `copy()`.

```kotlin
data class Badge(val text: String, val selected: Boolean = false)
```

Two `Badge` objects with the same constructor-property values compare equal with `==`. That does not mean they are the same instance. Properties declared only in the class body are not included in these generated comparisons or in the generated `copy` parameters. Put the values that define the record in the primary constructor.

In [ ]:
data class Badge(val text: String, val selected: Boolean = false)
val firstBadge = Badge("Work")
val equalBadge = Badge("Work")
println(firstBadge == equalBadge)
println(firstBadge)

The output is `true` and `Badge(text=Work, selected=false)`. Kotlin generated the value comparison and readable text from the constructor properties. An ordinary class does not automatically gain this data-class equality behavior.

### Copy with a Deliberate Change

A data class's **copy operation** creates a new instance while letting you replace selected constructor-property values. Omitted arguments retain the original values. This uses the named/default-argument ideas from Lesson 2.

```kotlin
val chosenBadge = firstBadge.copy(selected = true)
```

The new object keeps `text` and changes `selected`. It does not change `firstBadge`. Both objects can have `val` properties because creating a new value is different from assigning to an existing property.

In [ ]:
val chosenBadge = firstBadge.copy(selected = true)
println("Original: ${firstBadge.selected}")
println("Copy: ${chosenBadge.selected}")

This prints `Original: false` and `Copy: true`. The copy receives a changed constructor value; the original object stays as it was.

`copy()` is a **shallow copy**: it does not recursively duplicate objects stored inside properties. If a property referred to a mutable nested object, both copies could still refer to that same nested object. Our text and Boolean properties avoid that shared-mutation issue. A data class is not automatically deeply immutable.

### Promise Behavior with an Interface

An **interface** states behavior that an implementing class must provide. It is useful when callers should depend on an operation rather than a particular implementation. The `interface` keyword declares the contract; a colon after a class name lists what it implements.

```kotlin
interface LabelSource {
    fun label(): String
}
class FixedLabel(val text: String) : LabelSource {
    override fun label(): String = text
}
```

The interface requires a `label` method returning non-null text. `FixedLabel` provides that method. The `override` keyword makes this implementation explicit. There are no constructor-call parentheses after `LabelSource`, because an interface has no constructor to call. The next example stores the implementation through the interface type.

In [ ]:
interface LabelSource {
    fun label(): String
}
class FixedLabel(val text: String) : LabelSource {
    override fun label(): String = text
}
val labelSource: LabelSource = FixedLabel("Ready")
println(labelSource.label())

The result is `Ready`. The variable's type promises the interface operation, while the actual object supplies its implementation. A different class could implement `LabelSource` with another rule without changing a caller that needs only `label()`.

### Inheritance Must Be Explicit

Kotlin classes and methods are closed to inheritance or overriding by default. The `open` keyword permits it. An overriding method must use `override`. This makes extension points visible when reading the parent class.

In the next example, `PriorityStatus` extends `StatusLabel`. The parentheses in `StatusLabel()` invoke the superclass constructor, unlike the interface name above. The subclass supplies a different implementation of the open method.

In [ ]:
open class StatusLabel {
    open fun text(): String = "Ready"
}
class PriorityStatus : StatusLabel() {
    override fun text(): String = "Priority"
}
val status: StatusLabel = PriorityStatus()
println(status.text())

This prints `Priority`: the object's overriding method is used. Removing `open` from the parent class or its method would reject this inheritance or override at compile time. Use inheritance for a meaningful relationship, not merely to save a few lines. Interfaces often express a narrower behavior contract.

<details class="animation-panel" open>
<summary>Follow an original note and its copy — show or hide animation</summary>
<p><img src="media/03_classes_properties_and_app_data/copy_note.gif" alt="A note starts with archived false. A copy request creates another note with archived true. The original remains false." width="960" style="max-width:100%;height:auto;"></p>
</details>

The diagram applies the copy rule to the Note data class used next. The original and copied object remain distinct; only the copy receives the changed archive value. The 10-second animation loops; closing its panel hides the motion.

[View the labeled still diagram](media/03_classes_properties_and_app_data/copy_note_still.png).

## Worked Example

### Represent and Format a Note

A note needs a title and an archive flag. Creating an archived version should preserve the original. A formatter should promise to return a note's display text.

1. `Note` puts both properties in its primary constructor, so the generated copy operation includes them.
2. `NoteFormatter` declares one required method. Its `note` parameter uses the new `Note` type just as earlier functions used `String`.
3. `TitleFormatter` implements that method with an expression body that returns `note.title`.
4. `originalNote.copy(archived = true)` creates the changed version. The original binding and original properties remain intact.
5. A formatter object receives the copy. The last two prints compare the two archive values explicitly.

Read the example as three separate roles: the data, the behavior contract, and the class that supplies that behavior.

In [ ]:
data class Note(val title: String, val archived: Boolean = false)
interface NoteFormatter {
    fun format(note: Note): String
}
class TitleFormatter : NoteFormatter {
    override fun format(note: Note): String = note.title
}
val originalNote = Note("Lab ideas")
val archivedNote = originalNote.copy(archived = true)
val formatter = TitleFormatter()
println(formatter.format(archivedNote))
println("Original archived: ${originalNote.archived}")
println("Copy archived: ${archivedNote.archived}")

The output is:

```text
Lab ideas
Original archived: false
Copy archived: true
```

The formatter prints the copied note's title, which was retained because the copy call changed only `archived`. The original remains unarchived. The final two lines are evidence that `copy` created a changed instance rather than mutating the original. The formatter does not change either object; it returns text from the object it receives.

## Guided Practice

### Trace a renamed copy

Before running the next cell, predict both output lines. Identify which property changes and which property keeps its existing value. Explain why the copied completion flag is true or false even though the constructor declares a default of false. Then run the cell to check.

In [ ]:
Your prediction:
Two output lines:
Changed and retained properties, with reasoning:


In [ ]:
data class PreviewReminder(val title: String, val completed: Boolean = false)
val currentReminder = PreviewReminder("Read chapter 1", completed = true)
val renamedReminder = currentReminder.copy(title = "Read chapter 2")
println("Original: ${currentReminder.title}, ${currentReminder.completed}")
println("Copy: ${renamedReminder.title}, ${renamedReminder.completed}")

<details>
<summary>Show answer</summary>

```text
Original: Read chapter 1, true
Copy: Read chapter 2, true
```

The call supplies a new title only. Generated `copy` parameters use the source object's current values for omitted arguments, so the completion flag remains true. The constructor's false default is used when constructing a new reminder without that argument; it does not reset an existing value during copying. The original object keeps both of its properties.

</details>

### Complete initialization, then change its rule

The starter below is valid but does not yet handle an empty title. Fill its `init` block so an empty title becomes `"Untitled"` and any other title stays unchanged. An `init` block can update a `var` just as a method can, using the `if` and assignment syntax you already know.

Create drafts with `""` and `"Plan lab"`, then print each through `display()`. Access the private title through that method. Finally, change the empty-title replacement to `"New note"` and rerun. Only the empty-title result should change.

In [ ]:
class GuidedDraft(private var title: String) {
    init {
        // TODO: Replace an empty title with "Untitled".
    }
    fun display(): String = title
}
// TODO: Create and display one empty-title draft and one "Plan lab" draft.

<details>
<summary>Show answer</summary>

```kotlin
class GuidedDraft(private var title: String) {
    init {
        if (title == "") {
            title = "Untitled"
        }
    }
    fun display(): String = title
}
println(GuidedDraft("").display())
println(GuidedDraft("Plan lab").display())
```

The output is `Untitled` followed by `Plan lab`. Construction initializes the property from the argument and then runs `init`. The condition changes only the empty title. Because the property is `private`, outside callers use the public `display()` method rather than accessing `title` directly.

Replacing `"Untitled"` with `"New note"` changes the first output line to `New note`; `Plan lab` stays unchanged. Keep `var` on this property because initialization reassigns it. A missing `val` or `var` would make the constructor argument a parameter rather than this stored property.

</details>

### Repair an override

This **intentionally faulty snippet** should print `Urgent reminder`, but it does not compile:

```kotlin
open class BaseReminderLabel {
    fun text(): String = "Reminder"
}
class UrgentReminderLabel : BaseReminderLabel() {
    override fun text(): String = "Urgent reminder"
}
println(UrgentReminderLabel().text())
```

Explain why making the class `open` is not enough. Then write a corrected version that keeps both classes and the overriding method. Do not remove `override` to hide the problem.

In [ ]:
Your diagnosis:
Why the override is rejected:
What must change and why:


In [ ]:
// TODO: Write and run the corrected base class, subclass, and print call.

<details>
<summary>Show answer</summary>

```kotlin
open class BaseReminderLabel {
    open fun text(): String = "Reminder"
}
class UrgentReminderLabel : BaseReminderLabel() {
    override fun text(): String = "Urgent reminder"
}
println(UrgentReminderLabel().text())
```

An open class allows subclasses, but its members are still final unless declared open or otherwise overridable. Adding `open` to the base `text` method permits the subclass's `override`. The base-class constructor call keeps its parentheses in the inheritance list. The corrected call prints `Urgent reminder` because the subclass supplies that implementation.

</details>

## Independent Practice

### Model a reminder and format its status

Build a `ReminderItem` data class with constructor properties `title: String` and `completed: Boolean = false`. Use `val` properties so this task changes a reminder by copying it.

Declare a `ReminderFormatter` interface with `format(reminder: ReminderItem): String`. Implement it in `StatusReminderFormatter`. Return `To do: ` plus the title when the flag is false, or `Done: ` plus the title when it is true. Keep printing outside the formatter so another screen could reuse its returned text.

Create an original `Read chapter` reminder using the default completion flag. Use `copy(completed = true)` to make a finished reminder. Store your formatter in a variable typed `ReminderFormatter`. Print the formatted original and copy, then print their completion flags to check that the original was preserved:

```text
To do: Read chapter
Done: Read chapter
Original completed: false
Copy completed: true
```

Test the other direction too: create `Sketch screen` with completion true, then copy it with completion false. The titles must stay `Sketch screen`; the formatted messages must be `Done: Sketch screen` and `To do: Sketch screen`, and the original must remain true. Use the same classes for both tests.

In [ ]:
// TODO: Declare ReminderItem, ReminderFormatter, and StatusReminderFormatter.
// TODO: Create, copy, format, and check both test pairs described above.

<details>
<summary>Show answer</summary>

One solution is:

```kotlin
data class ReminderItem(val title: String, val completed: Boolean = false)
interface ReminderFormatter {
    fun format(reminder: ReminderItem): String
}
class StatusReminderFormatter : ReminderFormatter {
    override fun format(reminder: ReminderItem): String {
        val status = if (reminder.completed) "Done" else "To do"
        return "$status: ${reminder.title}"
    }
}
val originalReminder = ReminderItem("Read chapter")
val finishedReminder = originalReminder.copy(completed = true)
val reminderFormatter: ReminderFormatter = StatusReminderFormatter()
println(reminderFormatter.format(originalReminder))
println(reminderFormatter.format(finishedReminder))
println("Original completed: ${originalReminder.completed}")
println("Copy completed: ${finishedReminder.completed}")

val finishedSketch = ReminderItem("Sketch screen", completed = true)
val reopenedSketch = finishedSketch.copy(completed = false)
val sketchFormatter: ReminderFormatter = StatusReminderFormatter()
println(sketchFormatter.format(finishedSketch))
println(sketchFormatter.format(reopenedSketch))
println("Original completed: ${finishedSketch.completed}")
println("Copy completed: ${reopenedSketch.completed}")
```

The data class stores the related properties and supplies `copy`. The named copy argument changes only completion; each title is retained. The formatter class implements the interface without constructor parentheses after the interface name. Its `override` fulfills the promised method signature.

For the first pair, the four lines match the target output. The second pair prints:

```text
Done: Sketch screen
To do: Sketch screen
Original completed: true
Copy completed: false
```

An interface-typed reference can call `format` because that behavior is part of the interface. The `if` expression selects a status and the function returns text; it does not change either reminder. Mutating the original or ignoring the object returned by `copy` would not produce the required original/copy pair.

</details>

### Explain the evidence from both pairs

After running your tests, identify the output that shows each original was preserved. Explain why testing both false-to-true and true-to-false copies is stronger than testing only the first direction. Also name the interface method used by the formatter variable.

In [ ]:
Your test explanation:
Evidence that each original was preserved:
What the reverse-direction test adds:
Method available through the interface:


<details>
<summary>Show answer</summary>

The first original still prints `Original completed: false`; the second still prints `Original completed: true`. Their titles also remain `Read chapter` and `Sketch screen` in the formatted output. The reverse-direction test would catch a solution that always sets completion true or always prints Done for copies. `ReminderFormatter` declares `format`, so that method can be called through the interface-typed variable even though `StatusReminderFormatter` provides its implementation.

</details>

## Summary

- Constructor parameters marked `val` or `var` become object properties. Kotlin constructor calls do not use `new`.
- Properties and methods can be public or restricted with `private`. An `init` block runs during construction.
- A data class generates value operations from its primary-constructor properties.
- `copy` creates a new instance. Omitted values come from the source object; nested objects are not recursively copied.
- An interface promises behavior, and `override` marks an implementation.
- Both a superclass and an overridable method must permit extension, commonly with `open`.

Next, we will put many data objects into collections and use small functions to select and transform them.

## Reflection

Without looking back, state what happens to properties you omit from a data class's `copy` call. Then describe an app edit where retaining the original object would be useful. Explain how a small interface could let two different displays format that data differently. No code is required.

In [ ]:
Your reflection:
Omitted copy properties:
App edit and reason to retain the original:
Shared formatting behavior for different displays:


<details>
<summary>Show answer</summary>

Omitted copy arguments use the source object's current property values. For example, a notes editor could keep the original note while a user previews a changed title. One formatter could return just the title for a compact list; another could include status for a detailed view. Both could implement the same `format` method. Creating a copy alone does not build an undo system; the app must still retain and choose the appropriate version.

</details>

## Supplemental Reading

- [Kotlin classes](https://kotlinlang.org/docs/classes.html) — primary constructors, constructor properties, and initialization blocks.
- [Kotlin data classes](https://kotlinlang.org/docs/data-classes.html) — generated equality and copy behavior, including shallow-copy limits.
- [Kotlin interfaces](https://kotlinlang.org/docs/interfaces.html) — contracts and their implementations.
- [Kotlin inheritance](https://kotlinlang.org/docs/inheritance.html) — open classes, open members, and overrides.